# Lab 8: Interactive Notebooks with ipywidgets

## Objectives

By the end of this lab, you should be able to:

- Use `@interact` to swap which variables drive a chart, without hand-editing code
- Combine a widget with brushing in one exploratory workflow
- Choose the right widget for the kind of variable a reader is controlling
- Give widgets and charts reader-friendly labels instead of raw column names
- Build widgets yourself and arrange them with `HBox`/`VBox`, instead of relying on `@interact`'s defaults
- Link two charts to the same control, so one choice updates both views
- Explain why widgets need a running notebook, and what that means for sharing your work

Everything here is a building block for the interactive section of your **Project 2 notebook**.

## Setup

We'll start with the **penguins** data from Lab 7, then spend most of the lab on the **gapminder** data, which comes built into Plotly: life expectancy, population, and GDP per capita for 142 countries, every five years from 1952 to 2007.

In [ ]:
# !pip install pandas plotly ipywidgets palmerpenguins

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
from palmerpenguins import load_penguins

penguins = load_penguins().dropna()
# parallel_coordinates colors by a number, so give each species a numeric code
penguins_pc = penguins.assign(species_code=penguins["species"].astype("category").cat.codes)

gap = px.data.gapminder()
gap.head()

## 1. Getting started with `@interact`

In Lab 7, every chart had its variables hard-coded in the cell. `ipywidgets.interact()` lets you wrap a plotting function so a dropdown, slider, or other control reruns that function automatically whenever the selection changes, without you touching the code.

In [ ]:
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

#TODO: use interact to create a scatter plot of the penguin measurements, with x and y axes chosen by the user from the numeric columns
@interact()

def scatter_by_choice(x="bill_length_mm", y="flipper_length_mm"):
    pass

Try every combination of `x` and `y` in the dropdowns above. Which pairs show the clearest separation between species? Which look like there's almost no relationship at all? Notice you didn't write a single new line of plotting code to check each one; the widget reruns the function for you.

### Widgets plus brushing

Let's combine a widget with a chart that supports brushing: a multi-select widget that controls which columns feed the parallel coordinates plot, so you can compare different variable combinations without editing code, then brush the result the way you did with parallel coordinates in Lab 7.

In [ ]:
all_numeric = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

@interact(
    chosen_vars=widgets.SelectMultiple(
        options=all_numeric, value=tuple(all_numeric), description="Variables"
    )
)
def parcoords_by_choice(chosen_vars):
    if len(chosen_vars) < 2:
        print("Select at least two variables.")
        return
    fig = px.parallel_coordinates(
        penguins_pc, dimensions=list(chosen_vars), color="species_code",
        color_continuous_scale=px.colors.qualitative.Plotly[:3],
        title="Parallel coordinates (variables selected above)"
    )
    fig.show()

Hold Ctrl (or Cmd on Mac) to select multiple variables in the widget above. Try dropping `bill_depth_mm` and comparing the remaining three-axis view to the full four-axis one. Does removing a variable make the pattern easier or harder to read? Then brush an axis on whichever combination you land on, the same way you did in Lab 7.

## 2. Where `@interact` falls short

`@interact` gets you a working control in a few lines, but its defaults are built for you, the analyst, not for a reader. Here's another example, now with gapminder: the relationship between wealth and life expectancy, one year at a time.

In [ ]:
@interact(year=sorted(gap["year"].unique()))
def wealth_health(year):
    fig = px.scatter(
        gap[gap["year"] == year],
        x="gdpPercap", y="lifeExp", size="pop", color="continent",
        log_x=True,
    )
    fig.show()

<div class="alert alert-warning">
<b>Stop and think:</b>

Step through a few years in the dropdown. List everything that would confuse a reader who hasn't seen this data before. (Look at the control, the axes, the labels, and what happens to the chart as you change the year.)
</div>

Some things you may have noticed:

- A **dropdown** hides the fact that years are ordered; you can't "scrub" through time.
- The **axes rescale** every time, so a country can look like it moved when it didn't. You can't compare years.
- The labels are **raw column names** (`gdpPercap`, `lifeExp`).
- There's **no title** telling the reader what to look for.

The rest of this lab fixes each of these.

## 3. Choose the control for the variable

The control should match the kind of variable the reader is choosing, the same way an encoding should match the data type:

| The reader is choosing... | Data type | Good controls |
|---|---|---|
| One option from a few unordered categories | Nominal | `Dropdown`, `RadioButtons`, `ToggleButtons` |
| One value from an ordered sequence (years, ranks) | Ordinal / quantitative | `IntSlider`, `SelectionSlider` |
| A range of values | Quantitative | `IntRangeSlider`, `FloatRangeSlider` |
| Several categories at once | Nominal | `SelectMultiple`, a set of `Checkbox`es |
| On/off | Binary | `Checkbox`, `ToggleButton` |

A few settings worth knowing:

- `description=` puts a label next to the control.
- `continuous_update=False` on a slider redraws only when the reader lets go, instead of on every pixel of the drag. Use it whenever redrawing is slow.
- `step=` makes a slider stop only at values that exist in the data.

<div class="alert alert-info">
<b>Talk to a neighbor:</b>

For each question below, which widget would you give the reader, and why?

1. "How did life expectancy change in *my* continent?"
2. "Which countries had a GDP per capita between \$5,000 and \$20,000 in 2007?"
3. "Should the chart show population on a log scale or not?"
</div>

In [ ]:
# TODO: make a year slider that stops only at the years in the data (1952, 1957, ..., 2007),
# starts at 2007, and only redraws when the reader lets go
year_slider = widgets.IntSlider(

)
year_slider

## 4. Labels a reader can understand

Two places raw column names leak through:

- **Widget options.** If you pass a list of `(label, value)` pairs, the reader sees the label and your function receives the value.
- **The chart.** Use Plotly's `labels=` to rename axes and legends, and `hover_data=` with a format string to make hover text readable (`":,"` adds thousands separators, `":.1f"` rounds to one decimal).

To stop the chart jumping around between years:

- Fix the axes with `range_x=` and `range_y=`, so every year is drawn on the same scale.
- Fix the colors with `category_orders=`. Otherwise Plotly hands out colors in the order categories first appear in the data, so a continent can change color when the year changes.

In [ ]:
# Reader sees the label; the function receives the column name
y_dropdown = widgets.Dropdown(
    options=[("Life expectancy (years)", "lifeExp"),
             ("Population", "pop")],
    value="lifeExp",
    description="Show",
)

LABELS = {"gdpPercap": "GDP per capita (US$, log scale)", "lifeExp": "Life expectancy (years)",
          "pop": "Population", "continent": "Continent", "country": "Country"}
ORDER = {"continent": ["Africa", "Americas", "Asia", "Europe", "Oceania"]}

def wealth_vs(year, y):
    fig = px.scatter(
        gap[gap["year"] == year],
        x="gdpPercap", y=y, color="continent", hover_name="country",
        # TODO: format hover text: population with commas, GDP and life expectancy to 1 decimal
        hover_data={"pop": "", "gdpPercap": "", "lifeExp": "", "continent": False},
        labels=LABELS,
        log_x=True, log_y=(y == "pop"),
        # TODO: fix both axis ranges and the continent colors so they don't change between years
        range_x=[],
        range_y=[] if y == "lifeExp" else [],
        category_orders=,
        title=f"Wealth and {LABELS[y].lower()} by country, {year}",
    )
    fig.show()

interact(wealth_vs, year=year_slider, y=y_dropdown);

<div class="alert alert-warning">
<b>Stop and think:</b>

Slide from 1952 to 2007 with the axes fixed. What happens to the two clusters of countries over time? Could you have seen that with the axes rescaling every year?

Note the y-axis does change when you switch the dropdown between life expectancy and population, because they're different variables. That's fine: the reader asked for a different chart.
</div>

## 5. Building the layout yourself

`@interact` stacks every control above the chart in whatever order it likes. For a notebook someone else reads, you'll usually want to arrange things yourself. The pattern has three steps:

1. **Make the widgets** (you've done this).
2. **Connect them to a function** with `widgets.interactive_output(function, {"argument": widget, ...})`. This returns an *output area* where the chart appears; it reruns the function whenever any widget changes.
3. **Lay it out** with `widgets.HBox([...])` (side by side) and `widgets.VBox([...])` (stacked), then `display()` it.

Here we put both controls on one row above the chart, and add a continent filter.

In [ ]:
continent_dropdown = widgets.Dropdown(
    options=["All"] + sorted(gap["continent"].unique()),
    value="All",
    description="Continent",
)

def life_by_year(year, continent):
    data = gap[gap["year"] == year]
    if continent != "All":
        data = data[data["continent"] == continent]
    fig = px.scatter(
        data, x="gdpPercap", y="lifeExp", color="continent", hover_name="country",
        hover_data={"pop": ":,", "gdpPercap": ":,.1f", "lifeExp": ":.1f", "continent": False},
        labels=LABELS, log_x=True, range_x=[200, 120000], range_y=[20, 90], category_orders=ORDER,
        title=f"Wealth and life expectancy, {continent if continent != 'All' else 'all continents'}, {year}",
    )
    fig.show()

# TODO: connect life_by_year to year_slider and continent_dropdown
chart = widgets.interactive_output(life_by_year, {})

# TODO: put the two controls side by side, above the chart
display(widgets.VBox([

]))

Notice that `year_slider` is the **same widget object** you made in Section 3, and it now appears in two places. Move it in one place and it moves in the other. That's the idea behind the next section.

## 6. Linked views: one control, two charts

A **linked view** is two or more charts driven by the same control, so one choice updates them all. It lets the reader see the same selection from two angles at once.

Below, a single continent dropdown drives:

- **Left:** life expectancy over time for each country in that continent (the trend), and
- **Right:** the ten countries with the highest life expectancy in that continent in 2007 (the ranking).

Each chart gets its own `interactive_output`, but both are connected to the **same** widget.

In [ ]:
region = widgets.Dropdown(
    options=sorted(gap["continent"].unique()), value="Americas", description="Continent",
)

def trend(continent):
    data = gap[gap["continent"] == continent]
    fig = px.line(
        data, x="year", y="lifeExp", line_group="country", hover_name="country",
        labels=LABELS | {"year": "Year"}, range_y=[20, 90],
        title=f"Life expectancy over time, {continent}", width=520, height=420,
    )
    fig.update_traces(line_color="#9aa5b1")
    fig.show()

def ranking(continent):
    top = (gap[(gap["continent"] == continent) & (gap["year"] == 2007)]
           .nlargest(10, "lifeExp").sort_values("lifeExp"))
    fig = px.bar(
        top, x="lifeExp", y="country", orientation="h",
        labels=LABELS, range_x=[0, 90],
        title=f"Highest life expectancy, {continent}, 2007", width=520, height=420,
    )
    fig.show()

# TODO: connect BOTH charts to the same `region` widget, then show the dropdown above
# the two charts placed side by side
left =
right =
display(

)

<div class="alert alert-info">
<b>Talk to a neighbor:</b>

1. Why are all the lines on the left drawn in the same grey, with no legend? When would you want color instead?
2. Is this pair of charts better as a linked view, or would two separate widgets (one per chart) be better? Why?
</div>

## 7. Sharing a notebook with widgets

Widgets only work while a **Python kernel is running**: when the reader moves a slider, Python reruns your function. That has consequences for Project 2:

- An **HTML export** keeps your markdown and any chart that was showing, but the widgets won't respond. That's why Project 2 asks for the `.ipynb` file, not just HTML.
- Your notebook must **run top to bottom** in a fresh kernel (**Kernel → Restart Kernel and Run All Cells**) so someone else can bring the widgets back to life.
- Pick **good default values** (`value=`) for every widget: the default view is the first thing every reader sees, and the only thing they see in an export.
- Each widget-driven chart should come with a sentence saying **what the reader can do with it** and what to look for.

## On your own

Build a small interactive section, ideally with **your Project 2 data** (or with `gap` if your data isn't ready yet):

1. At least **one widget you built yourself**, chosen to fit the variable it controls, with reader-friendly labels.
2. A chart connected with `interactive_output` and laid out with `HBox`/`VBox`, with fixed axes where that helps comparison.
3. **Optional:** a second chart linked to the same widget.

Then, in the markdown cell below, write the 1–2 sentence note you'd put above this chart in your Project 2 notebook: what can the reader do with the control, and what should they look for?

In [ ]:
# TODO: Complete challenge here


**Type your 1–2 sentence reader note here**

## Submission

Widgets don't work in an HTML export, so for this lab submit **both** files on Canvas:

- your `.ipynb` file, and
- an HTML export (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar), so we can read your written answers.

Before you export, run **Kernel → Restart Kernel and Run All Cells** to make sure everything runs in order.